# PolyWin R2 — v13: GBM trio stack + MT-GNN per-target blend (kernel)

## Protocol (honest, self-contained)
* Reads only the active-round data: **train.csv / test.csv / PI1M.csv** (no archive, no
  uploaded artifacts).
* **Level 0** — two independent arms, verbatim from `mt_gnn_v2.py` and bit-identical to the
  0.8632-local-CV run:
  1. **GBM trio stack** — per-target LightGBM + XGBoost + CatBoost fold-safe OOF fed to a
     fold-consistent `Ridge(alpha=1.0)`.
  2. **MT-GNN** — a PI1M-inflected GINE trunk with per-target heads + leak-safe cross-target
     twin features, trained with GroupKFold-on-canonical folds (a canon's rows never train a
     model that predicts another row of the same canon).
* **Pretraining** — the GINE encoder is pretrained on the unlabeled PI1M corpus inside this
  run (masked atom/bond reconstruction), then the MT-GNN fine-tunes fold-safely. No external
  checkpoint is read.
* **Blend (v13)** — per-target `Ridge(alpha=1.0)` on the [GBM-stack, MT-GNN] OOF/test pairs,
  exactly the recipe in `v13_blend.py` that scored 0.8632 locally vs 0.8435 (GBM-only) /
  0.8382 (GNN-only). Small targets lean GNN, big targets lean GBM.
* All seeds fixed (`SEED = 42`); folds are GroupKFold by canonical SMILES.

Only OSI-approved libs: PyTorch, PyG, RDKit, scikit-learn, LightGBM, CatBoost, XGBoost.


In [ ]:
import os, sys, time, gc, random, warnings
import subprocess, importlib.util

def ensure_pkg(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print("installing", pkg, flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--disable-pip-version-check", pkg])

for _p, _n in [("rdkit", "rdkit"), ("torch_geometric", "torch_geometric"),
               ("lightgbm", "lightgbm"), ("catboost", "catboost"), ("xgboost", "xgboost")]:
    ensure_pkg(_p, _n)

# --- Pre-import CUDA fix: torch_geometric install may have pulled a torch that
#     has no kernel for the allocated GPU ("no kernel image"). Probe in a clean
#     subprocess BEFORE importing torch; if it fails, reinstall the broad-CUDA
#     cu121 wheel so the kernel below picks it up. ---
_probe = ('import torch;' + 'a=torch.zeros(4,device="cuda");b=a+1;torch.cuda.synchronize();print("OK")')
def _force_cuda():
    try:
        _r = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                            text=True, timeout=600)
    except Exception:
        _r = None
    if _r is not None and _r.returncode == 0 and "OK" in (_r.stdout or ""):
        return
    print("CUDA kernel missing for this GPU; installing torch 2.5.1 (cu121, supports sm_60)...", flush=True)
    try:
        # Pin 2.5.1: unlike the base cu128 build (sm_70+) it ships sm_60 kernels for
        # the allocated Tesla P100, and it stays compatible with numpy 2.0.2. Do NOT
        # touch numpy/pandas (a force-reinstall corrupts the numpy C ABI symbols).
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--no-cache-dir", "--index-url",
                               "https://download.pytorch.org/whl/cu121", "torch==2.5.1"],
                              timeout=1800)
        _r2 = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                             text=True, timeout=600)
        print("post-reinstall probe rc:", _r2.returncode,
              "out:", (_r2.stdout or "").strip(), "err:", (_r2.stderr or "")[-200:], flush=True)
    except Exception as _e:
        print("torch reinstall errored:", repr(_e)[:200], flush=True)
if os.path.exists("/kaggle"):
    _force_cuda()

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from rdkit.Chem import Descriptors, AllChem, MACCSkeys, rdMolDescriptors, Crippen, GraphDescriptors
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
GNN_SEEDS = os.environ.get("GNN_SEEDS", "42,999,2025")
os.environ["GNN_SEEDS"] = GNN_SEEDS
print("GNN_SEEDS =", GNN_SEEDS, flush=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
def _cuda_ok():
    if not torch.cuda.is_available():
        return False
    try:
        a = torch.zeros(4, device="cuda"); b = a + 1; torch.cuda.synchronize(); del a, b
        return True
    except Exception:
        return False
DEVICE = "cuda" if _cuda_ok() else "cpu"
print("device:", DEVICE, flush=True)
GLOBAL_FOLDS = 5
MAX_EPOCHS = 120
PATIENCE = 20
EARLY_HOLDOUT = 0.15
BS = 256
LR = 1e-3
if DEVICE == "cpu":
    GLOBAL_FOLDS = min(GLOBAL_FOLDS, 2)
    MAX_EPOCHS = min(MAX_EPOCHS, 12)
    BS = 256
    PATIENCE = min(PATIENCE, 6)
PRETRAIN_EPOCHS = 5
PRETRAIN_SAMPLE = 20000

if os.path.exists("/kaggle"):
    WORK = "/kaggle/working"; INP = "/kaggle/input"
else:
    WORK = os.path.join("vault", "pipeline_out_v13")
    INP = "official_dataset"
os.makedirs(WORK, exist_ok=True)
PRETRAINED = os.path.join(WORK, "pretrained_encoder.pt")
OUT = WORK

print("device:", DEVICE, "| SMOKE:", SMOKE, "| folds:", GLOBAL_FOLDS,
      "| PRETRAIN_EPOCHS:", PRETRAIN_EPOCHS, "| out:", OUT, flush=True)


In [ ]:
from sklearn.linear_model import Ridge

TARGETS = ["eea", "egb", "egc", "ei", "eps", "nc", "tg"]
TARGET_IDX = {t: i for i, t in enumerate(TARGETS)}


## 1. Data — current-round CSVs, canonicalize, compute descriptors + fingerprints

In [ ]:
def find_input(base, name):
    for p in [os.path.join(base, name), os.path.join(base, "ppp-round-2", name),
              os.path.join(base, "competitions", "ppp-round-2", name)]:
        if os.path.exists(p):
            return p
    return None

def canonical(s):
    if not isinstance(s, str):
        return None, None, None
    m = Chem.MolFromSmiles(s)
    if m is None:
        return None, None, None
    try:
        c = Chem.MolToSmiles(m)
        ik = Chem.MolToInchiKey(m)
    except Exception:
        return Chem.MolToSmiles(m), None, None
    return c, ik, m

def feats(m):
    if m is None:
        return [np.nan] * 22
    return [
        Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
        Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m),
        Descriptors.RingCount(m), Descriptors.NumAromaticRings(m),
        Descriptors.NumAliphaticRings(m), Descriptors.NumSaturatedRings(m),
        Descriptors.NumRotatableBonds(m), rdMolDescriptors.CalcNumHeavyAtoms(m),
        Descriptors.NumHeteroatoms(m), Descriptors.FractionCSP3(m),
        Crippen.MolMR(m), rdMolDescriptors.CalcNumBridgeheadAtoms(m),
        rdMolDescriptors.CalcNumSpiroAtoms(m),
        rdMolDescriptors.CalcNumAromaticAtoms(m) if hasattr(rdMolDescriptors, "CalcNumAromaticAtoms") else Descriptors.NumAromaticRings(m),
        GraphDescriptors.BalabanJ(m), GraphDescriptors.Ipc(m),
        rdMolDescriptors.CalcNumLipinskiHBA(m), rdMolDescriptors.CalcNumLipinskiHBD(m),
        rdMolDescriptors.CalcNumAtomStereoCenters(m),
    ]

FNAMES = ["MolWt", "LogP", "TPSA", "HDon", "HAccep", "RingCnt", "AroRing", "AliRing", "SatRing",
          "RotB", "HeavyAt", "HeteroAt", "FracCSP3", "MR", "Bridge", "Spiro", "AroAt",
          "BalabanJ", "Ipc", "LipHBA", "LipHBD", "Stereo"]
assert len(FNAMES) == 22

train_path = find_input(INP, "train.csv")
test_path = find_input(INP, "test.csv")
assert train_path and test_path, "train.csv / test.csv not found in " + INP

tr = pd.read_csv(train_path)
te = pd.read_csv(test_path)
print("train:", tr.shape, "test:", te.shape, flush=True)

tcpl = tr["smiles"].map(canonical)
tr["canon"], tr["inchikey"], _ = zip(*tcpl)
tepl = te["smiles"].map(canonical)
te["canon"], te["inchikey"], _ = zip(*tepl)

tr_f = np.array(tr["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
te_f = np.array(te["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
tr[FNAMES] = tr_f
te[FNAMES] = te_f
print("descriptors done", flush=True)

# Drop rows whose canonical SMILES could not be parsed -> keep clean.
trf = tr.dropna(subset=["target"]).copy()
tef = te.copy()

FEAT_COLS = [c for c in trf.columns if c not in
            ("smiles", "target", "target_type", "canon", "inchikey", "id")]
print("FEAT_COLS:", len(FEAT_COLS), flush=True)


In [ ]:
def add_fingerprints(df):
    morgan = np.zeros((len(df), 2048), dtype=np.float32)
    maccs = np.zeros((len(df), 167), dtype=np.float32)
    for i, s in enumerate(df["smiles"]):
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        morgan[i] = np.frombuffer(AllChem.GetMorganFingerprintAsBitVect(
            m, 2, nBits=2048).ToBitString().encode(), "u1") - ord("0")
        maccs[i] = np.frombuffer(MACCSkeys.GenMACCSKeys(m).ToBitString().encode(),
                                 "u1") - ord("0")
    return morgan, maccs

F32_MAX = np.finfo(np.float32).max

def clean_feats(df):
    D = np.clip(df[FEAT_COLS].values, -F32_MAX, F32_MAX)
    for j in range(D.shape[1]):
        col = D[:, j]
        med = np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0.0
        col[~np.isfinite(col)] = med
    return D.astype(np.float32)

D_tr = clean_feats(trf)
mor_tr, mc_tr = add_fingerprints(trf)
X = np.hstack([D_tr, mor_tr, mc_tr]).astype(np.float32)
Xs = StandardScaler().fit(X).transform(X).astype(np.float32)

D_te = clean_feats(tef)
mor_te, mc_te = add_fingerprints(tef)
Xte = np.hstack([D_te, mor_te, mc_te]).astype(np.float32)
Xtes = StandardScaler().fit(X).transform(Xte).astype(np.float32)

Y = trf["target"].values.astype(np.float32)
T = trf["target_type"].values
G = trf["canon"].values.astype(str)

idx_of_target = {t: np.where(T == t)[0] for t in TARGETS}
print("train:", X.shape, "test:", Xte.shape, "targets:", TARGETS, flush=True)


## 2. Level-0 sources (verbatim from mt_gnn_v2.py: graph feats + GINE + MT-GNN)

In [ ]:
# Graph featurization (MUST match the v10 pretrain kernel so the saved
# pretrained_encoder.pt loads into the same GINEEncoder).
# =====================================================================
ATOM_SYMBOLS = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "Si", "P", "OTHER"]
HYBRIDIZATIONS = ["SP", "SP2", "SP3", "SP3D", "SP3D2", "OTHER"]
BOND_TYPES = ["SINGLE", "DOUBLE", "TRIPLE", "AROMATIC"]


def one_hot(value, choices):
    vec = [0.0] * len(choices)
    idx = choices.index(value) if value in choices else len(choices) - 1
    vec[idx] = 1.0
    return vec


def atom_features(atom):
    return (one_hot(atom.GetSymbol(), ATOM_SYMBOLS)
            + one_hot(atom.GetHybridization().name, HYBRIDIZATIONS)
            + [atom.GetIsAromatic() * 1.0, atom.IsInRing() * 1.0,
               atom.GetDegree() / 4.0, atom.GetTotalNumHs() / 4.0,
               atom.GetFormalCharge() / 2.0])


N_ATOM_FEATS = len(ATOM_SYMBOLS) + len(HYBRIDIZATIONS) + 5
N_BOND_FEATS = len(BOND_TYPES) + 2


def bond_features(bond):
    return one_hot(bond.GetBondType().name, BOND_TYPES) + [
        bond.GetIsConjugated() * 1.0, bond.IsInRing() * 1.0]


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() < 2:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [bf, bf]
    if len(edge_index) == 0:
        edge_index = [[0, 0]]; edge_attr = [[0.0] * N_BOND_FEATS]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


def build_graphs(df, has_target=True):
    out = {}
    freq = df["target_type"].value_counts(normalize=True)
    for row_id, row in zip(df.index, df.itertuples()):
        g = smiles_to_graph(row.smiles)
        if g is None:
            continue
        g.row_id = row_id
        g.smiles = row.smiles
        if has_target:
            g.target_idx = torch.tensor([TARGET_IDX[row.target_type]], dtype=torch.long)
            g.y = torch.tensor([float(row.target)], dtype=torch.float)
            g.w = torch.tensor([1.0 / freq[row.target_type]], dtype=torch.float)
        out[row_id] = g
    return out


def to_pyg(graphs):
    if isinstance(graphs, dict):
        graphs = list(graphs.values())
    return Batch.from_data_list(graphs)


t0 = time.time()
train_graphs = build_graphs(trf, has_target=True)
test_graphs = build_graphs(tef, has_target=False)
print(f"graphs: {len(train_graphs)} train, {len(test_graphs)} test "
      f"({time.time()-t0:.0f}s)", flush=True)


# =====================================================================
# Shared encoder + multi-task trunk (same GINEEncoder as v10 kernel).
# =====================================================================
class GINEEncoder(nn.Module):
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4, dropout=0.2):
        super().__init__()
        self.atom_encoder = nn.Linear(n_atom_feats, hidden)
        self.bond_encoder = nn.ModuleList(
            [nn.Linear(n_bond_feats, hidden) for _ in range(n_layers)])
        self.convs = nn.ModuleList(); self.bns = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.bns.append(nn.BatchNorm1d(hidden))
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        h = self.atom_encoder(x)
        for conv, bn, bond_enc in zip(self.convs, self.bns, self.bond_encoder):
            e = bond_enc(edge_attr)
            h = conv(h, edge_index, e)
            h = bn(h); h = F.relu(h); h = F.dropout(h, p=self.dropout,
                                                    training=self.training)
        return h


class MTGNN(nn.Module):
    """Shared trunk + per-target heads. Optional cross-target twin features
    are concatenated to the pooled embedding before the shared trunk."""

    def __init__(self, n_atom_feats, n_bond_feats, n_twin=0, hidden=128,
                 n_layers=4, dropout=0.2):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden,
                                   n_layers, dropout)
        pool_in = hidden * 2 + n_twin
        self.trunk = nn.Sequential(
            nn.Linear(pool_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Dropout(dropout))
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout),
                          nn.Linear(64, 1))
            for _ in TARGETS])

    def forward(self, data, twin=None):
        h = self.encoder(data.x, data.edge_index, data.edge_attr)
        pooled = torch.cat([global_mean_pool(h, data.batch),
                            global_add_pool(h, data.batch)], dim=1)
        if twin is not None and twin.size(1) > 0:
            pooled = torch.cat([pooled, twin.to(pooled.device)], dim=1)
        ht = self.trunk(pooled)
        out = torch.empty(data.batch.max() + 1, len(TARGETS), device=h.device)
        for i, head in enumerate(self.heads):
            out[:, i] = head(ht)[:, 0]
        return out

    def load_encoder(self, state_dict):
        enc = {k[len("encoder."):]: v for k, v in state_dict.items()
               if k.startswith("encoder.")}
        missing, unexpected = self.encoder.load_state_dict(enc, strict=False)
        print(f"  encoder init: missing={len(missing)} unexpected={len(unexpected)}",
              flush=True)


# =====================================================================

## 3. Pretrain the GINE encoder on PI1M (in-kernel self-supervised)

In [ ]:
pl_path = find_input(INP, "PI1M.csv")
pl = []
if pl_path:
    pldf = pd.read_csv(pl_path)
    smi_col = "SMILES" if "SMILES" in pldf.columns else "smiles"
    pldf = pldf[[smi_col]].rename(columns={smi_col: "smiles"})
    pldf["canon"] = pldf["smiles"].map(lambda s: canonical(s)[0])
    pldf = pldf.dropna(subset=["canon"])
    pl = pldf.drop_duplicates("canon")["smiles"].tolist()
    rng = np.random.RandomState(SEED); rng.shuffle(pl)
    pl = pl[:PRETRAIN_SAMPLE]
    print("PI1M pretraining corpus:", len(pl), "SMILES (capped at", PRETRAIN_SAMPLE, ")", flush=True)
else:
    print("no PI1M: pretraining skipped", flush=True)

def build_pretrain_graphs(smiles_list):
    graphs = []
    for smi in smiles_list:
        g = smiles_to_graph(smi)
        if g is not None:
            graphs.append(g)
    return graphs

pl_graphs = build_pretrain_graphs(pl) if pl else []
print("pretraining graphs:", len(pl_graphs), flush=True)

from torch_geometric.loader import DataLoader

class PretrainedEncoder(nn.Module):
    # wraps the shared GINE trunk; state_dict keys start with 'encoder.'
    # so MTGNN.load_encoder can load them (identical to the reference kernel).
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4,
                 mask_atom=0.15, mask_bond=0.20):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden, n_layers)
        self.atom_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_atom_feats))
        self.bond_proj = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_bond_feats))
        self.mask_atom = mask_atom; self.mask_bond = mask_bond

    def forward(self, x, edge_index, edge_attr, batch):
        n = x.size(0); m = edge_index.size(1)
        atom_mask = torch.rand(n, device=x.device) < self.mask_atom
        bond_mask = torch.rand(m, device=x.device) < self.mask_bond
        x_c = x.clone(); x_c[atom_mask] = 0.0
        ea_c = edge_attr.clone(); ea_c[bond_mask] = 0.0
        h = self.encoder(x_c, edge_index, ea_c)
        if atom_mask.any():
            atom_loss = F.mse_loss(self.atom_proj(h[atom_mask]), x[atom_mask])
        else:
            atom_loss = torch.zeros((), device=x.device)
        if bond_mask.any():
            src = h[edge_index[0, bond_mask]]
            dst = h[edge_index[1, bond_mask]]
            if src.numel() > 0:
                bond_loss = F.mse_loss(self.bond_proj(torch.cat([src, dst], dim=1)), edge_attr[bond_mask])
            else:
                bond_loss = torch.zeros((), device=x.device)
        else:
            bond_loss = torch.zeros((), device=x.device)
        return atom_loss, bond_loss

def pretrain(epochs=PRETRAIN_EPOCHS, batch_size=256, lr=1e-3):
    if not pl_graphs:
        print("No PI1M graphs - pretraining skipped", flush=True)
        return None
    model = PretrainedEncoder(N_ATOM_FEATS, N_BOND_FEATS).to(DEVICE)
    loader = DataLoader(pl_graphs, batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best = np.inf; best_state = None; t0 = time.time()
    for epoch in range(epochs):
        model.train(); tot_a = 0.0; tot_b = 0.0; nbl = 0
        for batch in loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            a_loss, b_loss = model(batch.x, batch.edge_index, batch.edge_attr, batch)
            loss = a_loss + 0.5 * b_loss
            loss.backward(); opt.step()
            tot_a += a_loss.item(); tot_b += b_loss.item(); nbl += 1
            del batch
        va = (tot_a + 0.5 * tot_b) / max(nbl, 1)
        if va < best:
            best = va; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"pretrain ep {epoch+1}/{epochs}: loss={va:.4f} ({time.time()-t0:.0f}s)", flush=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if best_state:
        torch.save(best_state, PRETRAINED)
        print("saved pretrained_encoder.pt", flush=True)
    return best_state

print("=== Pretraining GNN on PI1M ===", flush=True)
pretrained_state = pretrain()


## 4. Level-0 predictions (verbatim: leak-safe twins + MT-GNN fold OOF + GBM trio stack)

In [ ]:
# Twin source: per-target LGBM OOF (leak-safe) + fold-bagged test preds.
# twin_u(row i) = target-u LGBM's prediction on row i's features.
# =====================================================================
print("\n=== Twin source: per-target LGBM OOF (leak-safe) ===", flush=True)
lgb_test_te = np.zeros((len(Xte), len(TARGETS)), dtype=np.float32)
TARGET_MEAN = {t: float(Y[idx_of_target[t]].mean()) for t in TARGETS}


# For train, twin_u(row i) uses lgb_oof_all from the target-u LGBM. But
# lgb_oof_all is stored by row index only for target-u rows. For a row of
# target t, its target-u twin value is the OOF prediction of the model_u on
# THAT row's features - we approximate with the per-target model_u evaluated
# on every train row (OOF where available, fold-safe holdout elsewhere).
# Simplest leak-safe approach: evaluate each target-u LGBM on ALL train rows
# via a dedicated OOF-style pass below.
print("\n=== Building leak-safe twin feature matrices ===", flush=True)
twin_train = np.zeros((len(X), (len(TARGETS) - 1) * 2), dtype=np.float32)
twin_test = np.zeros((len(Xte), (len(TARGETS) - 1) * 2), dtype=np.float32)
col_map = {}
for u in TARGETS:
    col = 0
    for t2 in TARGETS:
        if t2 == u:
            continue
        col_map[(u, t2)] = (col, col + 1)
        col += 2
def leak_safe_oof_scores():
    """For each target u, score every train row with a model trained on a
    canon-group that excludes that row (grouped OOF across all targets)."""
    scores = np.full((len(X), len(TARGETS)), np.nan, dtype=np.float32)
    # canon -> group id per row, using one global fold assignment
    gkf = GroupKFold(n_splits=GLOBAL_FOLDS)
    row_fold = np.zeros(len(X), dtype=int)
    for f, (_, va) in enumerate(gkf.split(Xs, Y, G)):
        row_fold[va] = f
    for u in TARGETS:
        for f in range(GLOBAL_FOLDS):
            in_fold = np.where(row_fold == f)[0]
            out_fold = np.setdiff1d(np.arange(len(X)), in_fold)
            idx_u_out = np.intersect1d(out_fold, idx_of_target[u])
            if len(idx_u_out) == 0:
                continue
            fit_ids, ho_ids = train_test_split(idx_u_out,
                                               test_size=EARLY_HOLDOUT,
                                               random_state=SEED)
            m = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05,
                                  num_leaves=15, min_child_samples=10,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=SEED, verbose=-1)
            m.fit(Xs[fit_ids], Y[fit_ids], eval_set=[(Xs[ho_ids], Y[ho_ids])])
            scores[in_fold, TARGET_IDX[u]] = m.predict(Xs[in_fold])
            # test bag
            lgb_test_te[:, TARGET_IDX[u]] += m.predict(Xtes) / GLOBAL_FOLDS
    return scores, lgb_test_te


twin_scores, lgb_test_te = leak_safe_oof_scores()
for t in TARGETS:
    for u in TARGETS:
        if u == t:
            continue
        iu = TARGET_IDX[u]
        c0, c1 = col_map[(t, u)]
        impute = TARGET_MEAN[u]
        v = twin_scores[:, iu]
        miss = np.isnan(v).astype(np.float32)
        v = np.where(miss, impute, v)
        twin_train[:, c0] = v; twin_train[:, c1] = miss
        # test: fold-bagged model_u prediction, always available
        tv = lgb_test_te[:, iu]
        tmiss = np.isnan(tv).astype(np.float32)
        tv = np.where(tmiss, impute, tv)
        twin_test[:, c0] = tv; twin_test[:, c1] = tmiss
print("twin matrices:", twin_train.shape, twin_test.shape, flush=True)


# =====================================================================
# MT-GNN fold-safe OOF + test bag
# =====================================================================
def early_split(fit_ids):
    uniq_g = np.unique(G[fit_ids])
    uniq_f, uniq_h = train_test_split(uniq_g, test_size=EARLY_HOLDOUT,
                                      random_state=SEED)
    return (fit_ids[np.isin(G[fit_ids], uniq_f)],
            fit_ids[np.isin(G[fit_ids], uniq_h)])


row_to_graph = {g.row_id: g for g in train_graphs.values()}
print("\n=== MT-GNN v2 (pretrained-init trunk + twins) ===", flush=True)
pretrained_state = torch.load(PRETRAINED, map_location="cpu") if os.path.exists(
    PRETRAINED) else None
if pretrained_state is not None:
    print("loaded pretrained_encoder.pt", flush=True)

GNN_SEEDS = [int(s) for s in os.environ.get("GNN_SEEDS", "42").split(",") if s.strip()]


def run_gnn_seed(seed):
    """One seed's MT-GNN: fold-safe GroupKFold OOF + fold-bagged test preds.
    Returns (mt_oof_all, mt_test) in raw scale. Identical math to the v13 run
    except torch/np/random seeding are reset per seed."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    n_twin = twin_train.shape[1]
    mt_oof_all = np.full(len(X), np.nan, dtype=np.float32)
    mt_test_folds = np.zeros((len(Xte), GLOBAL_FOLDS), dtype=np.float32)
    for f, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=GLOBAL_FOLDS).split(
            Xs, Y, G)):
        t0f = time.time()
        stats = {}
        y_norm = np.empty(len(tr_idx), dtype=np.float32)
        for t in TARGETS:
            mask = (T[tr_idx] == t)
            if mask.sum() > 0:
                mu, sd = Y[tr_idx][mask].mean(), Y[tr_idx][mask].std() + 1e-6
                stats[t] = (mu, sd)
                y_norm[mask] = (Y[tr_idx][mask] - mu) / sd
        fit_ids, ho_ids = early_split(tr_idx)
        model = MTGNN(N_ATOM_FEATS, N_BOND_FEATS, n_twin=n_twin).to(DEVICE)
        if pretrained_state is not None:
            model.load_encoder(pretrained_state)
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        pos_of = {int(o): p for p, o in enumerate(tr_idx)}
        pos_of_all = {int(o): p for p, o in enumerate(tr_idx)}

        def predict_ids(ids, m=model):
            m.eval()
            out = np.empty(len(ids), dtype=np.float32)
            with torch.no_grad():
                for i in range(0, len(ids), 256):
                    bi = ids[i:i + 256]
                    graphs = [row_to_graph[int(b)] for b in bi]
                    batch = to_pyg(graphs).to(DEVICE)
                    twin = torch.tensor(twin_train[bi], dtype=torch.float)
                    p = m(batch, twin=twin).cpu().numpy()
                    for j, b in enumerate(bi):
                        ti = TARGET_IDX[T[b]]
                        mu, sd = stats[T[b]]
                        out[i + j] = p[j, ti] * sd + mu
            return out

        best, best_r2, pat = None, -np.inf, 0
        for ep in range(MAX_EPOCHS):
            model.train()
            perm = np.random.permutation(len(fit_ids))
            for i in range(0, len(perm), BS):
                bi = fit_ids[perm[i:i + BS]]
                idxs = [pos_of_all[int(b)] for b in bi]
                yb = torch.tensor(y_norm[idxs]).unsqueeze(1).to(DEVICE)
                wb = torch.tensor([row_to_graph[int(b)].w.item() for b in bi],
                                  dtype=torch.float).unsqueeze(1).to(DEVICE)
                graphs = [row_to_graph[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_train[bi], dtype=torch.float)
                opt.zero_grad()
                pred = model(batch, twin=twin)
                ti = torch.tensor([TARGET_IDX[T[b]] for b in bi], device=DEVICE)
                pred_sel = pred.gather(1, ti.unsqueeze(1))
                loss = (F.mse_loss(pred_sel, yb, reduction="none") * wb).mean()
                loss.backward(); opt.step()
            hp = predict_ids(ho_ids)
            hr = r2_score(Y[ho_ids], hp)
            if hr > best_r2:
                best_r2 = hr
                best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                pat = 0
            else:
                pat += 1
                if pat >= PATIENCE:
                    break
        model.load_state_dict(best)
        mt_oof_all[va_idx] = predict_ids(va_idx)
        # test prediction via graphs
        model.eval()
        with torch.no_grad():
            te_pred = np.zeros(len(Xte), dtype=np.float32)
            for i in range(0, len(Xte), 256):
                bi = np.arange(i, min(i + 256, len(Xte)))
                graphs = [test_graphs[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_test[bi], dtype=torch.float)
                p = model(batch, twin=twin).cpu().numpy()
                for j, b in enumerate(bi):
                    ttt = tef["target_type"].iloc[int(b)]
                    ti = TARGET_IDX[ttt]
                    mu, sd = stats[ttt]
                    te_pred[i + j] = p[j, ti] * sd + mu
        mt_test_folds[:, f] = te_pred
        print(f"seed {seed}  fold {f}: holdout R2={best_r2:.4f} ({time.time()-t0f:.0f}s)", flush=True)
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    assert not np.isnan(mt_oof_all).any()
    return mt_oof_all, mt_test_folds.mean(axis=1)


print("GNN_SEEDS =", GNN_SEEDS, flush=True)
mt_oof_sum = np.zeros(len(X), dtype=np.float32)
mt_test_sum = np.zeros(len(Xte), dtype=np.float32)
for _gs in GNN_SEEDS:
    _oo, _mt = run_gnn_seed(_gs)
    mt_oof_sum += _oo
    mt_test_sum += _mt
mt_oof_all = mt_oof_sum / len(GNN_SEEDS)
mt_test = mt_test_sum / len(GNN_SEEDS)
assert not np.isnan(mt_oof_all).any()

mt_oof = {t: mt_oof_all[idx_of_target[t]] for t in TARGETS}


# =====================================================================
# Per-target fallback vs the GBM trio stack (Ridge on lgb+xgb+cb).
# =====================================================================
print("\n=== GBM trio stack OOF (fallback floor) ===", flush=True)
gbm_oof = {t: {m: np.zeros(len(idx_of_target[t])) for m in ('lgb', 'xgb', 'cb')}
           for t in TARGETS}
gbm_test = {t: {m: np.zeros(len(Xte)) for m in ('lgb', 'xgb', 'cb')} for t in TARGETS}
import xgboost as xgb
import catboost as cb

for t in TARGETS:
    idx = idx_of_target[t]
    Xt, yt, gt = Xs[idx], Y[idx], G[idx]
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(Xt, yt, gt):
        fit_ids, ho_ids = early_split(tr_idx)
        l = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03,
                              num_leaves=15, min_child_samples=10, subsample=0.8,
                              colsample_bytree=0.8, random_state=SEED, verbose=-1)
        x = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=4,
                             subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                             random_state=SEED, verbosity=0)
        c = cb.CatBoostRegressor(iterations=2000, learning_rate=0.03, depth=6,
                                 random_seed=SEED, task_type='CPU', verbose=False,
                                 allow_writing_files=False)
        for m, est in ((l, l), (x, x), (c, c)):
            est.fit(Xt[fit_ids], yt[fit_ids], eval_set=[(Xt[ho_ids], yt[ho_ids])])
        gbm_oof[t]['lgb'][va_idx] = l.predict(Xt[va_idx])
        gbm_oof[t]['xgb'][va_idx] = x.predict(Xt[va_idx])
        gbm_oof[t]['cb'][va_idx] = c.predict(Xt[va_idx])
        gbm_test[t]['lgb'] += l.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['xgb'] += x.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['cb'] += c.predict(Xtes) / GLOBAL_FOLDS
    print(f"  {t} done", flush=True)

from sklearn.linear_model import Ridge

stack_oof = {}
stack_test = {}
for t in TARGETS:
    idx = idx_of_target[t]
    yt = Y[idx]; gt = G[idx]
    M = np.column_stack([gbm_oof[t][m] for m in ('lgb', 'xgb', 'cb')])
    Mte = np.column_stack([gbm_test[t][m] for m in ('lgb', 'xgb', 'cb')])
    oof = np.zeros(len(idx)); te_pred = np.zeros(len(Xte))
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(M, yt, gt):
        r = Ridge(alpha=1.0).fit(M[tr_idx], yt[tr_idx])
        oof[va_idx] = r.predict(M[va_idx])
        te_pred += r.predict(Mte) / GLOBAL_FOLDS
    stack_oof[t] = oof; stack_test[t] = te_pred

## 5. v13 blend — per-target Ridge (OOF-tuned alpha) on [GBM, MT-GNN] OOF + submission

In [ ]:
ALPHA_GRID = [0.1, 0.5, 1.0, 2.5, 5.0, 10.0, 25.0]

oof_gbm_global = np.full(len(X), np.nan, dtype=np.float32)
oof_mt_global = np.full(len(X), np.nan, dtype=np.float32)
for t in TARGETS:
    idx = idx_of_target[t]
    oof_gbm_global[idx] = stack_oof[t]
    oof_mt_global[idx] = mt_oof[t]
assert not np.isnan(oof_gbm_global).any() and not np.isnan(oof_mt_global).any()

test_gbm_global = np.zeros(len(Xte), dtype=np.float32)
test_mt_global = np.zeros(len(Xte), dtype=np.float32)
for t in TARGETS:
    m_te = (tef["target_type"] == t).values
    test_gbm_global[m_te] = stack_test[t][m_te]
    test_mt_global[m_te] = mt_test[m_te]

print("\n=== Check 1: corr(GBM, GNN) per target ===", flush=True)
for t in TARGETS:
    idx = idx_of_target[t]
    c = np.corrcoef(oof_gbm_global[idx], oof_mt_global[idx])[0, 1]
    print(f"  {t:<4} corr={c:.4f}", flush=True)

print("\n=== tuning per-target Ridge alpha (OOF-selected) ===", flush=True)
rows = []
coefs = {t: [] for t in TARGETS}
best_a = {}
final_te = np.zeros(len(tef))
for t in TARGETS:
    idx = idx_of_target[t]
    yt = Y[idx].astype(np.float64)
    Mx = np.column_stack([oof_gbm_global[idx], oof_mt_global[idx]])
    Mte = np.column_stack([test_gbm_global, test_mt_global])
    # 1) pick alpha by OOF R2 (nested, fold-safe)
    cv = list(GroupKFold(n_splits=GLOBAL_FOLDS).split(Mx, yt, G[idx]))
    oof_r2 = {}
    for a in ALPHA_GRID:
        o = np.zeros(len(idx))
        for trk, vk in cv:
            o[vk] = Ridge(alpha=a).fit(Mx[trk], yt[trk]).predict(Mx[vk])
        oof_r2[a] = r2_score(yt, o)
    a_best = max(oof_r2, key=oof_r2.get)
    best_a[t] = a_best
    # 2) final OOF (for report) + test bag with the chosen alpha
    oof = np.zeros(len(idx)); te_pred = np.zeros(len(tef))
    for trk, vk in cv:
        lr = Ridge(alpha=a_best); lr.fit(Mx[trk], yt[trk])
        oof[vk] = lr.predict(Mx[vk])
        te_pred += lr.predict(Mte) / GLOBAL_FOLDS
        coefs[t].append(lr.coef_.tolist())
    m_te = (tef["target_type"] == t).values
    final_te[m_te] = te_pred[m_te]
    r_blend = r2_score(yt, oof); r_g = r2_score(yt, oof_gbm_global[idx]); r_m = r2_score(yt, oof_mt_global[idx])
    cb = np.mean(coefs[t], axis=0)
    rows.append(dict(target=t, alpha=float(a_best), blend=r_blend, GBM=r_g, GNN=r_m,
                     w_GBM=cb[0], w_GNN=cb[1]))
    print(f"  {t:<4} alpha={a_best:<6} blend={r_blend:.4f} GBM={r_g:.4f} GNN={r_m:.4f} "
          f"w_GBM={cb[0]:.3f} w_GNN={cb[1]:.3f}", flush=True)

# persist fold OOF / test preds so offline blend experiments don't rerun the GNN
np.savez(os.path.join(OUT, "blend_oof_test.npz"),
         oof_gbm=oof_gbm_global, oof_mt=oof_mt_global,
         test_gbm=test_gbm_global, test_mt=test_mt_global,
         y_all=Y.astype(np.float64), g_all=G.astype(str), t_all=T.astype(str))
print("wrote blend_oof_test.npz", flush=True)

df = pd.DataFrame(rows).set_index("target")
print("\n=== summary ===", flush=True)
print("  mean blend=%.4f | GBM=%.4f | GNN=%.4f | delta-vs-GBM %+.4f" % (
    df["blend"].mean(), df["GBM"].mean(), df["GNN"].mean(),
    df["blend"].mean() - df["GBM"].mean()), flush=True)

print("\n=== Check 2: blend weights (small targets should lean GNN) ===", flush=True)
for t in TARGETS:
    cb = np.mean(coefs[t], axis=0)
    print(f"  {t:<4} alpha={best_a[t]:.2f} w_GBM={cb[0]:.3f} w_GNN={cb[1]:.3f} GNN_share={cb[1]/(cb.sum()):.2f}", flush=True)

sub = pd.DataFrame({"id": tef["id"].values, "target": final_te})
sub_path = os.path.join(OUT, "submission_v13.csv")
sub.to_csv(sub_path, index=False)
print("\nwrote", sub_path, flush=True)
print("  rows", len(sub), "| NaN", sub["target"].isna().sum(),
      "| range [%.2f, %.2f]" % (sub["target"].min(), sub["target"].max()), flush=True)
df.round(4).to_csv(os.path.join(OUT, "v13_blend_report.csv"), index=True)
print("wrote", os.path.join(OUT, "v13_blend_report.csv"), flush=True)
print("DONE", flush=True)
